In [20]:
import pandas as pd
import numpy as np
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler

In [21]:
# 1. carregas o preços limpos gerados no notebook 01
PRECOS_B3_PATH = f'../data/processed/stocks_b3.csv'
df_precos = pd.read_csv(PRECOS_B3_PATH, index_col=0, parse_dates=True)

# 2. calcular os retornos diários (a variação percentual entre um dia e outro)
df_retornos = df_precos.pct_change().dropna()

print("=== RETORNOS DIÁRIOS (AMOSTRA) === ")
print(df_retornos.head())

=== RETORNOS DIÁRIOS (AMOSTRA) === 
            USIM5.SA  BAZA3.SA  KLBN4.SA  B3SA3.SA  EGIE3.SA  RENT3.SA  \
Date                                                                     
2011-07-19 -0.004273  0.066667 -0.001919 -0.003144 -0.004556 -0.032432   
2011-07-20 -0.006009 -0.041667  0.003846  0.012619 -0.016018 -0.009976   
2011-07-21  0.039724  0.000000  0.003832  0.035306 -0.011628  0.060057   
2011-07-22 -0.034053  0.021739  0.030534 -0.012037  0.027451  0.004182   
2011-07-25 -0.012038 -0.021276  0.018519  0.002031 -0.023664  0.012117   

            GOAU4.SA  CMIG4.SA  LREN3.SA  DXCO3.SA  ...  BBDC4.SA  ABEV3.SA  \
Date                                                ...                       
2011-07-19 -0.011892 -0.006899 -0.004293  0.030756  ...  0.014235  0.000000   
2011-07-20 -0.016412 -0.006514  0.036551 -0.019608  ...  0.012281 -0.006097   
2011-07-21  0.039488  0.011474  0.024955  0.035652  ...  0.024263 -0.002454   
2011-07-22 -0.018192 -0.006483  0.010057 -0.002519

In [22]:
# 1. retorno esperado pelo ano (252 dias úteis)
retorno_esperado = df_retornos.mean() * 252

# 2. volatilidade atualizada
volatilidade = df_retornos.std() * np.sqrt(252)

df_metricas = pd.DataFrame({
    'Retorno_Esperado': retorno_esperado,
    'Volatilidade': volatilidade
})

print("\n=== METRICAS === ")
print(df_metricas.head())


=== METRICAS === 
          Retorno_Esperado  Volatilidade
USIM5.SA          0.142535      0.546441
BAZA3.SA          0.172895      0.361470
KLBN4.SA          0.174245      0.293324
B3SA3.SA          0.213395      0.369584
EGIE3.SA          0.143334      0.236597


In [23]:
### CÁLCULO DO BETA (O RISCO DE MERCADO)
# beta mede a sensibilidade da ação em frente ao índice Ibovespa
# beta = 1 (move de acordo com o mercado), Beta > 1 (mais agressiva), Beta < 1 (mais defensiva)

# 1. baixa ibovespa para o mesmo período dos dados usados
data_inicio = df_retornos.index.min().strftime('%Y-%m-%d')
data_fim = df_retornos.index.max().strftime('%Y-%m-%d')

print(f"\nbaixando Ibovespa de {data_inicio} a {data_fim}...")
ibov = yf.download('^BVSP', start=data_inicio, end=data_fim)
ibov_close = ibov['Close']

# 2. calcula o retorno diário do Ibovespa
ibov_retornos = pd.DataFrame(ibov_close.pct_change().dropna())
ibov_retornos.columns = ['IBOV']

# 3. alinha as datas, junta os retornos das ações com o do IBOV para concidir os dias
df_alinhado = df_retornos.join(ibov_retornos, how='inner')

# 4. calcula o beta para cada ação
betas = {}
variancia_ibov = df_alinhado['IBOV'].var()

for acao in df_retornos.columns:
    # beta = covariancia(acao, mercado) / variancia(mercado)
    covariancia = df_alinhado[[acao, 'IBOV']].cov().iloc[0,1]
    beta = covariancia / variancia_ibov
    betas[acao] = beta

df_metricas['Beta'] = pd.Series(betas)

print("\n=== MÉTRICAS COM BETA CALCULADO ===")
print(df_metricas.head())


baixando Ibovespa de 2011-07-19 a 2026-07-13...


[*********************100%***********************]  1 of 1 completed


=== MÉTRICAS COM BETA CALCULADO ===
          Retorno_Esperado  Volatilidade      Beta
USIM5.SA          0.142535      0.546441  1.301059
BAZA3.SA          0.172895      0.361470  0.400722
KLBN4.SA          0.174245      0.293324  0.465474
B3SA3.SA          0.213395      0.369584  1.144641
EGIE3.SA          0.143334      0.236597  0.536452


In [24]:
# ==========================================
# BLOCO 4: ÍNDICE DE RISCO DO ATIVO (IRA)
# ==========================================
# A Intuição: Combinar Volatilidade e Beta para criar um índice único de risco
# escalonado de 1 a 5, compatível semanticamente com o IPR (Índice de Perfil de Risco).

# 1. Calcular o Índice Sharpe (mantido apenas para validação posterior)
TAXA_LIVRE_RISCO = 0.105
df_metricas['Sharpe'] = (df_metricas['Retorno_Esperado'] - TAXA_LIVRE_RISCO) / df_metricas['Volatilidade']

# 2. Calcular o IRA bruto (combinação linear de Volatilidade e Beta)
# Atribuímos pesos iguais (0.5) para as duas métricas de risco primárias.
df_metricas['IRA_raw'] = (0.5 * df_metricas['Volatilidade']) + (0.5 * df_metricas['Beta'])

# 3. Padronização
# Traduzimos o IRA_raw para a escala ANBIMA (1 a 5)
scaler_ira = MinMaxScaler(feature_range=(1, 5))
df_metricas['IRA'] = scaler_ira.fit_transform(df_metricas[['IRA_raw']])

# Criar o DataFrame final selecionando apenas as colunas necessárias
df_vetor_financeiro = df_metricas[['Retorno_Esperado', 'Volatilidade', 'Beta', 'Sharpe', 'IRA']].copy()
df_vetor_financeiro.index.name = 'Ticker'

print("\n=== VETOR FINANCEIRO FINAL (COM IRA) ===")
print(df_vetor_financeiro.head())

# 4. Guardar os dados para a Fase 4
df_vetor_financeiro.to_csv('../data/processed/b3_vetor_financeiro.csv')
print("\nVetor Financeiro das Ações guardado com sucesso em data/processed/!")


=== VETOR FINANCEIRO FINAL (COM IRA) ===
          Retorno_Esperado  Volatilidade      Beta    Sharpe       IRA
Ticker                                                                
USIM5.SA          0.142535      0.546441  1.301059  0.068689  4.554254
BAZA3.SA          0.172895      0.361470  0.400722  0.187831  1.777656
KLBN4.SA          0.174245      0.293324  0.465474  0.236069  1.768973
B3SA3.SA          0.213395      0.369584  1.144641  0.293289  3.701620
EGIE3.SA          0.143334      0.236597  0.536452  0.162020  1.805435

Vetor Financeiro das Ações guardado com sucesso em data/processed/!
